# 02 - confidence scheduler의 속도/안정성 절충

**학습 목표**: threshold가 낮을수록 한 forward에서 더 많은 token을 조기 확정하고, 높을수록 더 많은 round가 필요한 toy 과정을 만듭니다.

**실행 방법**: Python 3/Jupyter에서 cell을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 기능만 사용합니다.

수치는 합성 실험이며 논문 TPS를 재현하지 않습니다.

In [ ]:
# set은 아직 확정되지 않은 위치의 삭제·잔여 검사를 간단히 표현하기 위해 사용합니다.
def simulate(threshold, n_tokens=24):
    unresolved = set(range(n_tokens))
    rounds = 0
    commits = []
    premature = 0
    while unresolved:
        rounds += 1
        chosen = []
        for i in sorted(unresolved):
            # denoising round마다 confidence가 증가하는 결정적 toy 값입니다.
            confidence = min(0.50 + 0.085 * rounds + 0.025 * (i % 5), 0.999)
            if confidence >= threshold:
                chosen.append((i, confidence))
        if not chosen:
            i = max(unresolved, key=lambda j: 0.50 + 0.085 * rounds + 0.025 * (j % 5))
            chosen = [(i, min(0.50 + 0.085 * rounds + 0.025 * (i % 5), 0.999))]
        for i, confidence in chosen:
            unresolved.remove(i)
            premature += confidence < 0.75
        commits.append(len(chosen))
    return {'threshold': threshold, 'rounds': rounds, 'avg_tpf': n_tokens / rounds,
            'premature': premature, 'commits': commits}

for threshold in (0.60, 0.80, 0.95, 0.99):
    print(simulate(threshold))

fast = simulate(0.60)
safe = simulate(0.95)
assert fast['rounds'] <= safe['rounds']
assert fast['avg_tpf'] >= safe['avg_tpf']


## 논문과 연결

논문에서는 threshold 0.95가 93%+ 정확도와 108.9 TPS의 절충점이고, 0.6은 90% 초과 정확도에서 164.8 TPS입니다. 여기서는 하드웨어 시간을 흉내 내지 않고 TPF와 조기 확정 위험의 방향만 확인합니다.